In [1]:
import random
import time
from collections.abc import Mapping
from typing import Any

import httpx
import pandas as pd

In [2]:
REFERER = "https://quote.eastmoney.com/center/gridlist.html"

USER_AGENTS = [
    # Windows +Google Chrome
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.7922.75 Safari/537.36",
    # Iphone
    "Mozilla/5.0 (iPhone; CPU iPhone OS 18_6_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/18.6 Mobile/15E148 Safari/604.1",
    # Android
    "Mozilla/5.0 (Linux; Android 15; Pixel 9) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.7922.75 Mobile Safari/537.36",
]

request_headers = {
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "zh-CN,zh;q=0.9",
    "Referer":REFERER,
    "User-Agent": random.choice(USER_AGENTS)
}

In [3]:
client = httpx.Client(
  headers=request_headers,
  timeout=httpx.Timeout(connect=5.0, read=10.0, write=10.0, pool=10.0),
  # 设置最高连接数，防止误发大量并发，这样超量的会进行等待，超过pool timeout就报错
  limits=httpx.Limits(max_keepalive_connections=0, max_connections=1, keepalive_expiry=5.0),
  trust_env=False,
  follow_redirects=False,
)

## https://push2.eastmoney.com/api/qt/clist/get 接口分析
**GET** Method
- "http://push2.eastmoney.com/api/qt/clist/get" 报错 RemoteProtocolError: Server disconnected without sending a response.
- "http://push2delay.eastmoney.com/api/qt/clist/get" 成功（akshare方式）
- "http://pushguest.eastmoney.com/api/qt/clist/get" 开发者工具找到的，偶尔出现同1的错误，不要连发                                                                       
- 
### Request Params

该接口为分页接口，通过 `pn` 控制页码、`pz` 控制每页返回数量，拉取全量数据时从 `pn=1` 开始逐页递增，直到累计数量达到 `data.total`、当前页为空或返回数量小于 `pz`。`fields` 决定返回哪些字段，类似sql的select语法。`fid` 指定排序字段，`po=1` 为降序、`po=0` 为升，例如 `fid=f3` 直接获取涨幅靠前的记录，通常使用 `fid=f12` 保持分页顺序稳定。`np`为`1` 时 `data.diff` 为数组（recommended），`2` 时为以序号为键的对象。`fltt=1` 返回供网页格式化的缩放值，`fltt=2` 直接返回小数，程序化请求通常使用 `2`。`invt=2` 是语义未公开的行情兼容参数，参考原样保留。`ut` 是客户端标识而不是账号认证，固定值`fa5fd1943c7b386f172d6893dbfba10b
`。普通 JSON API 请求可省略后续参数：`cb` 用于 JSONP 包装，`dect`、`wbp2u` 是网页内部参数，`_` 是防缓存时间戳。

`fs` 用于筛选证券范围。表达式中的逗号 `,` 表示 OR，空格表示 AND；空格在 URL 查询字符串中通常编码为 `+`，所以 `m:1 t:2` 与 `m:1+t:2` 等价；`!` 表示排除。各标记及常见取值见下表：`m` 表示市场或数据源，`t` 表示该市场下的证券类型，`s` 表示更细的子类型，`b` 表示预定义集合，`f`、`e` 表示附加筛选，`i` 用于直接指定行情 ID。

| 标记 | 常见取值或写法 | 含义与示例 |
| --- | --- | --- |
| `m`：沪深与板块 | `0`、`1`、`2`、`90` | `m:0` 为深市数据源，北交所也使用它并叠加 `t:81 s:2048`；`m:1` 为沪市；`m:2` 为东财单独划分的中证系列指数行情源，不是第三个交易所；`m:90` 为板块数据源。 |
| `m`：海外证券与指数 | `105`、`106`、`107`、`116`、`124`、`125`、`128`、`153`、`155`、`156`、`305` | `105`、`106`、`107` 分别用于纳斯达克、纽交所和美股第三子市场；`116` 仅见于 efinance 的港股市场号映射，当前 `fs` 预设使用 `128`；`124`、`125`、`305` 为港股指数子市场，`128` 为港股证券，`153` 为美股 OTC/Pink/ADR 类市场，`155` 为伦交所，`156` 为伦交所 IOB 等国际证券子市场。 |
| `m`：外汇 | `119`、`120`、`133` | 分别为普通外汇交叉盘、人民币中间价和离岸人民币交叉盘；常用组合为 `m:119,m:120,m:133`。 |
| `m`：期货 | `8`、`113`、`114`、`115`、`142`、`225` | 分别为中金所、上期所、大商所、郑商所、上海国际能源交易中心和广期所。 |
| `m`：期权 | `10`、`12`、`140`、`141`、`151`、`163`、`226` | 分别为上交所、深交所、大商所、郑商所、上期所、上海国际能源交易中心和广期所期权。 |
| `t`：`m:0` 下的类型 | `5`、`6`、`7`、`10`、`13`、`80`、`81` | `t:5` 为深证指数，`t:6` 为深市 A 股，`t:7` 为深市 B 股，`t:10 e:97` 为深市 REITs，`t:13` 为部分资金流接口的兼容分支，`t:80` 为创业板，`t:81 s:2048` 为北交所证券。 |
| `t`：`m:1` 下的类型 | `1`、`2`、`3`、`9`、`23` | `t:1` 为上证指数，`t:2` 为沪市 A 股，`t:3` 为沪市 B 股，`t:9 e:97` 为沪市 REITs，`t:23` 为科创板。 |
| `t`：其他市场 | `m:90 t:1/2/3`、`m:128 t:1/2/3/4` | 板块数据源下 `t:1`、`t:2`、`t:3` 分别为地域、行业、概念板块；港股数据源下 `t:1`、`t:2`、`t:3`、`t:4` 分别用于 REIT/信托、债务证券、主板和创业板。 |
| `t`：英股内部分类 | `m:155 t:1/2/3`、`m:156 t:1/2/5/6/7/8` | AkShare 和 efinance 将这些内部上市分段合并查询；源码没有给出稳定的一一中文名称。 |
| `s` | `2`、`3`、`2048` | `m:1 s:2` 和部分预设中的 `m:1 s:3` 用于上证指数分支；`m:0 s:3` 为两网及退市；`m:0 t:81 s:2048` 为北交所。`s` 必须结合 `m`、`t` 解释。 |
| `b`：`MK` 集合 | `MK0010`、`MK0021`—`MK0024`、`MK0827`、`MK0201`、`MK0216`—`MK0221`、`MK0354`、`MK0356`、`MK0404`—`MK0407` | 分别用于重要指数、ETF 子集合、中概股、知名美股分类、可转债、交易所回购和 LOF。`MKxxxx` 是不可继续拆解的内部集合 ID，不是证券代码。 |
| `b`：`BK` / `DLMK` 集合 | `BK0498`、`BK0707`、`BK0804`、`BKxxxx`、`DLMK0101`、`DLMK0106`、`DLMK0144`、`DLMK0146` | `BK` 用于板块或资格池，如 A/B 股比价、沪股通、深股通及动态板块；`DLMK` 用于网页预设列表，如 AH 股比价、知名港股和港股通成分池。 |
| `f` | `4`、`8`、`!2`、`!50` | `f:4` 为风险警示板，`f:8` 为新股；`f:!2`、`f:!50` 分别是资金流和板块相关调用中使用的排除条件，服务端标记名称未公开。 |
| `e` | `97` | 额外类别条件；`m:1 t:9 e:97,m:0 t:10 e:97` 用于沪深 REITs。 |
| `i` | `i:<市场号>.<代码>` | 直接指定行情 ID，例如 `i:1.000001`、`i:0.399001`、`i:100.HSI`、`i:100.SPX`；多个 ID 用逗号合并。 |

`fields` 常用字段解析：

| 字段 | 含义 |
| --- | --- |
| `f1` / `f152` | 价格类 / 比例类的精度与显示辅助字段，主要供 `fltt=1` 的网页格式化使用，不是独立行情指标。 |
| `f2` / `f3` / `f4` | 最新价 / 涨跌幅 / 涨跌额。 |
| `f5` / `f6` | 成交量 / 成交额。 |
| `f7` / `f8` | 振幅 / 换手率。 |
| `f9` / `f10` / `f11` | 动态市盈率 / 量比 / 5 分钟涨跌幅。 |
| `f12` / `f13` / `f14` | 证券代码 / 市场编号 / 证券名称；可将 `f13` 和 `f12` 的值拼成统一行情 ID。 |
| `f15` / `f16` / `f17` / `f18` | 最高价 / 最低价 / 今开价 / 昨收价。 |
| `f19` / `f26` | 证券类型内部编码 / 上市日期。 |
| `f20` / `f21` | 总市值 / 流通市值。 |
| `f22` / `f23` | 3 分钟涨速 / 市净率。 |
| `f24` / `f25` | 60 日涨跌幅 / 年初至今涨跌幅。 |
| `f28` / `f31` / `f32` | 昨结价 / 买一价 / 卖一价，常用于期货、期权或盘口行情。 |
| `f62` / `f124` | 主力净流入 / 交易时间。 |


In [4]:
target = "https://push2delay.eastmoney.com/api/qt/clist/get"

ETF_PAGINATION_PARAMS = {
  "np":1,
  "po":1,
  "fltt":2,
  "invt":2,
  "pn":1,
  "pz":5,
  "ut": "fa5fd1943c7b386f172d6893dbfba10b",
  "fs": "b:MK0021,b:MK0022,b:MK0023,b:MK0024,b:MK0827",
  "fid": "f12",
  "fields": "f12,f13,f14"
}

In [5]:
resp = client.request("GET", target, params=ETF_PAGINATION_PARAMS)
resp_json = resp.json()
data = resp_json.get("data", None)

In [6]:
print(f"当前市场ETF基金总数 {data.get('total', "unknown")}")

if data.get("diff", None):
    df = pd.DataFrame(data.get("diff"))
    print(df)
    

当前市场ETF基金总数 1594
      f12  f13          f14
0  589990    1  科创综指ETF华泰柏瑞
1  589980    1  科创100ETF汇添富
2  589960    1  科创新能源ETF易方达
3  589950    1   科创100ETF富国
4  589900    1    科创综指ETF博时


## https://push2his.eastmoney.com/api/qt/stock/kline/get 接口分析

**GET** Method

这个接口返回单个证券的历史 K 线。与前面的 `clist/get` 不同：`clist/get` 更适合证券列表或行情快照，`kline/get` 的核心结果在 `data.klines` 中，每一条记录是一串逗号分隔的 K 线字段。

### Request Params

| 参数 | 常见值 | 含义与用法 | `515080` 示例 |
| --- | --- | --- | --- |
| `secid` | `1.515080` | 东方财富行情 ID，格式为 `市场号.证券代码`；`1` 为沪市，`0` 为深市。`515080` 按 ETF 市场判断逻辑使用 `1.515080`。 | `1.515080` |
| `klt` | `1`、`5`、`15`、`30`、`60`、`101`、`102`、`103` | K 线周期：1/5/15/30/60 分钟，101 日，102 周，103 月。 | `101` |
| `fqt` | `0`、`1`、`2` | 复权：`0` 不复权，`1` 前复权，`2` 后复权。同步原始成交行情建议使用 `0`。 | `1`（前复权） |
| `beg` | `YYYYMMDD` | 开始日期；历史日 K 例如 `20240101`。 | `20240101` |
| `end` | `YYYYMMDD` | 结束日期；查询单日 K 时与 `beg` 写成同一天。 | `20260821` |
| `fields1` | `f1,f2,...` | 外层/辅助字段。AKShare 当前 ETF 封装使用 `f1`—`f6`，efinance 使用 `f1`—`f13`；通常不影响 `klines` 的列顺序。 | `f1,f2,f3,f4,f5,f6` |
| `fields2` | `f51,f52,...` | K 线字段选择，决定 `klines` 每行的字段顺序。 | f51日期时间，f52/53开盘/收盘，f54/55最高/最低，f56/57成交量/成交额，f58振幅，f59/60涨跌幅/额，f61换手率 |
| `rtntype` | `6` | efinance 当前请求中带有的返回类型参数；AKShare 当前 ETF 封装没有依赖它。直接调用时可保留 `6`，不要把它当作周期参数。 | `6` |
| `ut` | 一串固定字符串 | 网页客户端标识/兼容参数，不是账号认证。AKShare 当前实现带有该参数，普通请求可以沿用。 | `7eea...` |

`beg` 和 `end` 是日期范围参数；但要得到一根‘日 K’，还必须同时指定 `klt=101`。如果收盘后同步原始行情，推荐组合是 `klt=101 + fqt=0 + beg=目标日期 + end=目标日期`。如果需要复权行情，保持其他参数不变，只替换 `fqt`。

### 本示例返回的数据口径
下面的 `FIND_ONE_ETF_PARAMS` 明确设置了 `fqt=1`，因此 `resp_json["data"]["klines"]` 返回的是 `515080` 的**前复权日 K 线**，不是不复权行情。每行的字段顺序由 `fields2` 决定：`f51` 是日期，`f52`—`f55` 是开盘、收盘、最高、最低；本示例请求的这四个价格字段已经按 `fqt=1` 的前复权口径返回。成交量、成交额和换手率不是价格复权字段；涨跌幅、涨跌额等由价格计算的字段以接口返回值为准。
要获得同一标的的三种口径，需要分别请求 `fqt=0`（不复权）、`fqt=1`（前复权）和 `fqt=2`（后复权）；不能从当前这一份 `resp_json` 同时推导出另外两种口径。

In [7]:
KLINE_TARGET = "https://push2his.eastmoney.com/api/qt/stock/kline/get"
KLINE_COLUMNS = [
    "日期", "开盘", "收盘", "最高", "最低",
    "成交量", "成交额", "振幅", "涨跌幅", "涨跌额", "换手率",
]
EASTMONEY_ADJUSTMENT_LABELS = {
    0: "不复权",
    1: "前复权",
    2: "后复权",
}

FIND_ONE_ETF_PARAMS = {
        "fields1": "f1,f2,f3,f4,f5,f6",
        "fields2": "f51,f52,f53,f54,f55,f56,f57,f58,f59,f60,f61",
        "ut": "7eea3edcaed734bea9cbfc24409ed989",
        "rtntype": "6",
        "klt": 101,
        "fqt": 1,
        "beg": 20260817,
        "end": 20260821,
        "secid": "1.515080",
    }

In [8]:
resp = client.request("GET", KLINE_TARGET, params=FIND_ONE_ETF_PARAMS)
resp_json = resp.json()
resp_json

{'rc': 0,
 'rt': 17,
 'svr': 183119980,
 'lt': 1,
 'full': 0,
 'dlmkts': '',
 'dsc': '0',
 'data': {'code': '515080',
  'market': 1,
  'name': '中证红利ETF招商',
  'decimal': 3,
  'dktotal': 1617,
  'preKPrice': 1.555,
  'klines': ['2026-08-17,1.553,1.559,1.559,1.545,1857608,288762232.000,0.90,0.26,0.004,2.61',
   '2026-08-18,1.558,1.568,1.572,1.556,3061646,479436773.000,1.03,0.58,0.009,4.30',
   '2026-08-19,1.570,1.574,1.581,1.570,2776382,437783093.000,0.70,0.38,0.006,3.90',
   '2026-08-20,1.568,1.585,1.590,1.567,2283844,361232874.000,1.46,0.70,0.011,3.21',
   '2026-08-21,1.582,1.582,1.585,1.576,1648998,260770754.000,0.57,-0.19,-0.003,2.32']}}

In [9]:
fqt = FIND_ONE_ETF_PARAMS["fqt"]
print(
    f"本次请求返回：fqt={fqt}（{EASTMONEY_ADJUSTMENT_LABELS[fqt]}）"
)
eastmoney_kline_515080 = pd.DataFrame(
    [row.split(",") for row in resp_json["data"]["klines"]],
    columns=KLINE_COLUMNS,
)
eastmoney_kline_515080

本次请求返回：fqt=1（前复权）


,日期,开盘,收盘,最高,最低,成交量,成交额,振幅,涨跌幅,涨跌额,换手率
0,2026-08-17,1.553,1.559,1.559,1.545,1857608,288762232.000,0.90,0.26,0.004,2.61
1,2026-08-18,1.558,1.568,1.572,1.556,3061646,479436773.000,1.03,0.58,0.009,4.30
2,2026-08-19,1.570,1.574,1.581,1.570,2776382,437783093.000,0.70,0.38,0.006,3.90
3,2026-08-20,1.568,1.585,1.590,1.567,2283844,361232874.000,1.46,0.70,0.011,3.21
4,2026-08-21,1.582,1.582,1.585,1.576,1648998,260770754.000,0.57,-0.19,-0.003,2.32


## 公司行为接口：分红、送转和配股

`push2his.eastmoney.com` 的 K 线接口不返回完整的公司行为字段。本节使用东方财富数据中心的两个网页后端接口：`RPT_SHAREBONUS_DET` 获取分红、送股和转增，`RPT_IPO_ALLOTMENT` 获取配股。

东方财富字段中的现金和比例通常按“每10股”给出，需要归一化到每股。接口可能返回未来的除权日期，即使 `ASSIGN_PROGRESS` 已经是“实施分配”，因此实际事件还要同时满足 `EX_DIVIDEND_DATE <= AS_OF_DATE`。

### 请求参数说明

这两个请求共用同一个 GET 地址，`reportName` 决定查询哪一种报告以及返回字段。下面的取值是网页请求中使用的稳定写法；这个数据中心接口没有公开完整的参数枚举，因此没有实测或公开依据的值不当作正式协议。

| 参数 | 可用取值/写法 | 含义与注意事项 |
| --- | --- | --- |
| `reportName` | `RPT_SHAREBONUS_DET`；`RPT_IPO_ALLOTMENT` | 前者返回分红、送股、转增明细；后者返回配股明细。两种报告的字段集合不同，不能只替换代码而不替换报告名。 |
| `columns` | `ALL`；`字段名1,字段名2,...` | `ALL` 返回该报告全部字段；逗号分隔时只返回指定字段。字段名必须是对应报告实际存在的字段且区分报告 schema；无效字段不会静默忽略，而是通常 HTTP 200、`success=false`、业务码 `9501`。 |
| `source` | 推荐 `WEB`；也可省略 | 表示网页端数据来源标记。当前网页稳定使用 `WEB`；本次实测省略也能返回相同结果，但其他字符串没有公开、稳定的枚举，不建议依赖。 |
| `client` | 推荐 `WEB`；也可省略 | 表示网页客户端标记。当前网页稳定使用 `WEB`；本次实测省略也能返回相同结果，其他值即使偶尔被接受也不代表有稳定语义。 |
| `filter` | 省略；`(SECURITY_CODE="603888")`；`(SECURITY_CODE in ("600081","600082"))` | 过滤报告记录。六位股票代码应保留前导零并作为字符串；多个括号条件直接相邻表示 AND，`in (...)` 可表达同一字段的多个代码（OR）。省略时可遍历全报告；字段名或语法错误会返回业务失败。实测字符串值使用双引号，单引号可能解析失败。 |
| `pageNumber` | 从 `1` 开始的正整数 | 页码；`1` 是第一页。`0` 或负数在本次实测被服务端按第一页处理，但这不是应依赖的协议行为。 |
| `pageSize` | 正整数，建议 `1`—`500` | 每页条数。实测大于 `500` 时服务端仍最多返回 `500` 条；`0` 或负数会触发服务端的兜底行为，因此代码中应使用正整数。 |
| `sortColumns` | 返回字段名；多列用逗号分隔；也可省略 | 排序字段，例如 `EX_DIVIDEND_DATE`。字段名必须存在；多列排序要与 `sortTypes` 一一对应；省略或空值时使用服务端默认顺序。 |
| `sortTypes` | `1` 升序；`-1` 降序；多列用逗号分隔 | 与 `sortColumns` 逐项对应，例如 `sortColumns=SECURITY_CODE,EX_DIVIDEND_DATE` 配 `sortTypes=1,-1`。字段数不一致会返回业务码 `9501`；本 notebook 用 `-1` 让最新除权日排在前面。 |

当前参数字典使用 `columns="ALL"`，是为了避免漏掉复权计算所需字段：`RPT_SHAREBONUS_DET` 中重点关注 `SECURITY_CODE`、`EX_DIVIDEND_DATE`、`ASSIGN_PROGRESS`、`PRETAX_BONUS_RMB`、`BONUS_RATIO`、`IT_RATIO`；`RPT_IPO_ALLOTMENT` 中重点关注 `SECURITY_CODE`、`EX_DIVIDEND_DATE`、`PLACING_RATIO`、`ISSUE_PRICE`、`ISSUE_NUM` 和 `TOTAL_SHARES_BEFORE`。现金和送转/配股比例字段通常按每 10 股给出：`PRETAX_BONUS_RMB` 是每 10 股现金（元），`BONUS_RATIO`、`IT_RATIO`、`PLACING_RATIO` 是每 10 股对应的股数，代码再除以 10 归一化到每股；`ISSUE_PRICE` 是配股价（元/股）。这两个报告本身不提供官方除权参考价和除权前收盘价，归一化结果中的对应列保留为空，不能把报告里别的价格字段误当成它们。

In [10]:
EASTMONEY_DATA_CENTER_ENDPOINT = "https://datacenter-web.eastmoney.com/api/data/v1/get"
EASTMONEY_AS_OF_DATE = pd.Timestamp("2026-08-28")
EASTMONEY_CORPORATE_ACTION_HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://data.eastmoney.com/",
    "User-Agent": request_headers["User-Agent"],
}

EASTMONEY_SHARE_BONUS_PARAMS = {
    "reportName": "RPT_SHAREBONUS_DET",
    "columns": "ALL",
    "source": "WEB",
    "client": "WEB",
    "filter": '(SECURITY_CODE="603888")',
    "pageNumber": 1,
    "pageSize": 20,
    "sortColumns": "EX_DIVIDEND_DATE",
    "sortTypes": -1,
}

EASTMONEY_IPO_ALLOTMENT_PARAMS = {
    "reportName": "RPT_IPO_ALLOTMENT",
    "columns": "ALL",
    "source": "WEB",
    "client": "WEB",
    "filter": '(SECURITY_CODE="600081")',
    "pageNumber": 1,
    "pageSize": 20,
    "sortColumns": "EX_DIVIDEND_DATE",
    "sortTypes": -1,
}

In [11]:
share_bonus_resp = client.get(
    EASTMONEY_DATA_CENTER_ENDPOINT,
    params=EASTMONEY_SHARE_BONUS_PARAMS,
    headers=EASTMONEY_CORPORATE_ACTION_HEADERS,
)
share_bonus_resp_json = share_bonus_resp.json()

In [12]:
share_bonus_resp_json

{'version': 'af34ef8610aa592aa7cacdedbb7bfab6',
 'result': {'pages': 1,
  'data': [{'SECUCODE': '603888.SH',
    'SECURITY_NAME_ABBR': '新华网',
    'SECURITY_INNER_CODE': '1000326745',
    'ORG_CODE': '10091108',
    'SECURITY_CODE': '603888',
    'BONUS_IT_RATIO': 1,
    'BONUS_RATIO': 1,
    'IT_RATIO': None,
    'PRETAX_BONUS_RMB': 1.47,
    'PLAN_NOTICE_DATE': '2026-04-24 00:00:00',
    'EQUITY_RECORD_DATE': '2026-07-22 00:00:00',
    'EX_DIVIDEND_DATE': '2026-07-23 00:00:00',
    'REPORT_DATE': '2025-12-31 00:00:00',
    'ASSIGN_PROGRESS': '实施分配',
    'IMPL_PLAN_PROFILE': '10送1股派1.47元(含税,扣税后1.223元)',
    'NOTICE_DATE': '2026-07-16 00:00:00',
    'MARKET_TYPE': '069001001001',
    'EX_DIVIDEND_DAYS': 36,
    'IS_KCB': None,
    'BASIC_EPS': 0.4879,
    'BVPS': 5.575938372483,
    'PER_CAPITAL_RESERVE': 2.205885046331,
    'PER_UNASSIGN_PROFIT': 1.984903998835,
    'PNP_YOY_RATIO': 40.737047960802,
    'TOTAL_SHARES': 674738168,
    'PUBLISH_DATE': '2026-04-24 00:00:00',
    'DIVIDENT

In [13]:
ipo_allotment_resp = client.get(
    EASTMONEY_DATA_CENTER_ENDPOINT,
    params=EASTMONEY_IPO_ALLOTMENT_PARAMS,
    headers=EASTMONEY_CORPORATE_ACTION_HEADERS,
)
ipo_allotment_resp_json = ipo_allotment_resp.json()
ipo_allotment_resp_json

{'version': 'f9d8c4ef7024daaaf4a5eedf483db4cf',
 'result': {'pages': 1,
  'data': [{'FINANCE_CODE': '42976',
    'SECUCODE': '600081.SH',
    'SECURITY_CODE': '600081',
    'ORG_CODE': '10002305',
    'SECURITY_NAME_ABBR': '东风科技',
    'CORRECODE': '700081',
    'CORRECODE_NAME_ABBR': '东风配股',
    'PLACING_RATIO': 3,
    'ISSUE_PRICE': 9.59,
    'TOTAL_SHARES_BEFORE': 447276315,
    'ISSUE_NUM': 131067214,
    'TOTAL_SHARES_AFTER': 578343529,
    'EQUITY_RECORD_DATE': '2023-08-01 00:00:00',
    'PAY_START_DATE': '2023-08-02 00:00:00',
    'PAY_END_DATE': '2023-08-08 00:00:00',
    'LISTING_DATE': '2023-08-24 00:00:00',
    'EX_DIVIDEND_DATE': '2023-08-10 00:00:00',
    'TOTAL_RAISE_FUNDS': 1256934582.26,
    'NET_RAISE_FUNDS': 1251091688.67,
    'UNDERWRITE_WAY': '代销',
    'FIRST_NOTICE_DATE': '2023-07-28 00:00:00',
    'CHANGE_RATE': None},
   {'FINANCE_CODE': '1370025',
    'SECUCODE': '600081.SH',
    'SECURITY_CODE': '600081',
    'ORG_CODE': '10002305',
    'SECURITY_NAME_ABBR': '东风

## 公司行为 resp 字段 mapping

末尾两个请求中的 `share_bonus_resp` 和 `ipo_allotment_resp` 是 `httpx.Response`，解析后的 JSON 分别是 `share_bonus_resp_json` 和 `ipo_allotment_resp_json`。下面的 mapping 覆盖 HTTP 层字段、JSON 外壳字段，以及 `RPT_SHAREBONUS_DET` 和 `RPT_IPO_ALLOTMENT` 的每个 `result.data[]` 字段。`result.data` 是当前页的记录列表；字段值为 `None` 表示该记录不适用或接口没有提供，不应自动当成 0。

In [14]:
EASTMONEY_RESPONSE_FIELD_MAPPING = {
    "HTTP response (`*_resp`)": {
        "status_code": "HTTP 状态码；HTTP 200 不代表业务查询一定成功。",
        "headers": "HTTP 响应头，包含内容类型等传输元数据。",
        "content": "未经解析的原始响应字节。",
        "json()": "将响应体解析为 Python 字典；本接口正常返回 JSON。",
    },
    "common JSON envelope": {
        "version": "数据中心响应版本或版本 hash，用于后端版本/缓存标识。",
        "result": "结果容器；参数错误时可能为 null。",
        "result.pages": "按当前 filter、pageSize 计算出的总页数。",
        "result.data": "当前页的记录列表；每个元素对应一条公司行为记录。",
        "result.count": "当前 filter 命中的记录总数。",
        "success": "业务层是否成功；应与 HTTP 状态码一起检查。",
        "message": "业务状态或错误说明，例如成功时通常为 ok。",
        "code": "业务返回码；通常 0 表示成功，9501 常表示字段、过滤器或排序参数错误。",
    },
    "RPT_SHAREBONUS_DET.result.data[]": {
        "SECUCODE": "带市场后缀的证券代码，例如 603888.SH。",
        "SECURITY_NAME_ABBR": "证券简称。",
        "SECURITY_INNER_CODE": "东方财富内部证券标识，不是股票代码。",
        "ORG_CODE": "东方财富内部机构/公司标识。",
        "SECURITY_CODE": "六位股票代码，是目标股票代码字段。",
        "BONUS_IT_RATIO": "送转总比例，按每 10 股计；通常等于 BONUS_RATIO + IT_RATIO。",
        "BONUS_RATIO": "送股比例，按每 10 股计；例如 1 表示 10 送 1。",
        "IT_RATIO": "转增/转股比例，按每 10 股计；例如 0.5 表示 10 转 0.5。",
        "PRETAX_BONUS_RMB": "税前现金分红，单位为元/10 股；例如 1.47 表示 10 派 1.47 元。",
        "PLAN_NOTICE_DATE": "分红送转预案公告日。",
        "EQUITY_RECORD_DATE": "股权登记日。",
        "EX_DIVIDEND_DATE": "除权除息日；用于定位影响复权的实际发生日。",
        "REPORT_DATE": "分配方案所对应的财务报告期。",
        "ASSIGN_PROGRESS": "方案进度；实施分配表示已执行，其他值可能仍处于预案或审批阶段。",
        "IMPL_PLAN_PROFILE": "文字版实施分配方案，例如 10 送 1 股派 1.47 元。",
        "NOTICE_DATE": "最新公告日期，不一定等于预案公告日。",
        "MARKET_TYPE": "东方财富内部市场分类代码。",
        "EX_DIVIDEND_DAYS": "服务端计算的距除权除息日天数；未来日期可能为负数。",
        "IS_KCB": "科创板标识；具体值可能为空。",
        "BASIC_EPS": "基本每股收益，单位元/股。",
        "BVPS": "每股净资产，单位元/股。",
        "PER_CAPITAL_RESERVE": "每股资本公积，单位元/股。",
        "PER_UNASSIGN_PROFIT": "每股未分配利润，单位元/股。",
        "PNP_YOY_RATIO": "净利润同比增长率，原始值按百分数表示，例如 40.737 表示 40.737%。",
        "TOTAL_SHARES": "总股本，原始单位为股；网页展示时常换算为亿股。",
        "PUBLISH_DATE": "对应财务报告或分配资料的发布日期。",
        "DIVIDENT_RATIO": "接口字段名中的拼写为 DIVIDENT；股息率，原始值为小数，乘 100 才是百分数。",
        "D10_CLOSE_ADJCHRATE": "预案公告日后 10 日的收盘涨幅，单位百分数。",
        "BD10_CLOSE_ADJCHRATE": "股权登记日前 10 日的收盘涨幅，单位百分数。",
        "D30_CLOSE_ADJCHRATE": "除权除息日后 30 日的收盘涨幅，单位百分数。",
    },
    "RPT_IPO_ALLOTMENT.result.data[]": {
        "FINANCE_CODE": "东方财富内部融资/配股事项标识。",
        "SECUCODE": "带市场后缀的证券代码，例如 600081.SH。",
        "SECURITY_CODE": "六位股票代码，是目标股票代码字段。",
        "ORG_CODE": "东方财富内部机构/公司标识。",
        "SECURITY_NAME_ABBR": "证券简称。",
        "CORRECODE": "配股申购/配售代码。",
        "CORRECODE_NAME_ABBR": "配股申购/配售代码对应的简称。",
        "PLACING_RATIO": "配股比例，按每 10 股计；例如 3 表示 10 配 3。",
        "ISSUE_PRICE": "配股价格，单位元/股。",
        "TOTAL_SHARES_BEFORE": "配股前总股本，原始单位为股。",
        "ISSUE_NUM": "配股数量，单位股；以该报告的配股数量口径为准。",
        "TOTAL_SHARES_AFTER": "配股后总股本，原始单位为股。",
        "EQUITY_RECORD_DATE": "股权登记日。",
        "PAY_START_DATE": "配股缴款起始日期。",
        "PAY_END_DATE": "配股缴款截止日期。",
        "LISTING_DATE": "配股股份上市日期。",
        "EX_DIVIDEND_DATE": "配股除权日；用于定位影响复权的实际发生日。",
        "TOTAL_RAISE_FUNDS": "配股募集资金总额，单位元。",
        "NET_RAISE_FUNDS": "扣除发行费用后的募集资金净额，单位元；可能为空。",
        "UNDERWRITE_WAY": "承销/包销方式，例如代销、余额包销。",
        "FIRST_NOTICE_DATE": "首次公告日期。",
        "CHANGE_RATE": "配股页面使用的辅助涨跌/变动率字段；原始 schema 未说明明确计算基准，可能为空，不作为复权计算字段。",
    },
}

eastmoney_response_field_mapping = pd.DataFrame(
    [
        {
            "scope": scope,
            "field": field,
            "meaning": meaning,
        }
        for scope, fields in EASTMONEY_RESPONSE_FIELD_MAPPING.items()
        for field, meaning in fields.items()
    ]
)
eastmoney_response_field_mapping

,scope,field,meaning
0,HTTP response (`*_resp`),status_code,HTTP 状态码；HTTP 200 不代表业务查询一定成功。
1,HTTP response (`*_resp`),headers,HTTP 响应头，包含内容类型等传输元数据。
2,HTTP response (`*_resp`),content,未经解析的原始响应字节。
3,HTTP response (`*_resp`),json(),将响应体解析为 Python 字典；本接口正常返回 JSON。
4,common JSON envelope,version,数据中心响应版本或版本 hash，用于后端版本/缓存标识。
...,...,...,...
59,RPT_IPO_ALLOTMENT.result.data[],TOTAL_RAISE_FUNDS,配股募集资金总额，单位元。
60,RPT_IPO_ALLOTMENT.result.data[],NET_RAISE_FUNDS,扣除发行费用后的募集资金净额，单位元；可能为空。
61,RPT_IPO_ALLOTMENT.result.data[],UNDERWRITE_WAY,承销/包销方式，例如代销、余额包销。
62,RPT_IPO_ALLOTMENT.result.data[],FIRST_NOTICE_DATE,首次公告日期。
